# Prepare the Held-Out Stance Test Set

**Goal:** convert the held-out SemEval-2016 Task 6 gold test data (1249 individual
JSON files) into the same `conversations` format used by the training notebooks, then
push it to the HF Hub as its own dataset repo -- so it's loadable from anywhere
(including Colab, which has no access to your Mac's local files) for genuine
held-out evaluation, instead of improvising a held-out split from a training
subsample.

**This is an eval set, not training data.** It's the same gold test set the full
Snellius pipeline already evaluates against via the Tilburg vLLM pipeline --
confirmed zero overlap with the training rows.

Self-contained on purpose, same as the other notebooks: prompt/label logic defined
inline below, not imported from `prepare_stance_data.py`.


## 1. Define the prompt template + label map

Identical to the other notebooks -- must match exactly, since this is what the
fine-tuned model is actually evaluated against.


In [1]:
STANCE_PROMPT_TEMPLATE = (
    "Stance classification is the task of determining the expressed or implied opinion, "
    "or stance, of a document toward a certain, specified target. "
    "Analyze the following document and determine its stance toward the provided query.\n\n"
    "QUERY: {target}\n\n"
    "DOCUMENT: {text}\n\n"
    'Return valid JSON in exactly this format: {{"stance": "FAVOR"}}\n'
    'The "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\n'
    'Use "FAVOR" only when the author is definitely in favor of the query. '
    'Use "AGAINST" only when the author is definitely against the query. '
    'Use "NONE" if any of the following holds: (a) the document does not discuss the query '
    "at all, (b) the document discusses it but the author takes no clear side "
    "(neutral/balanced), or (c) the author's position cannot be determined with confidence. "
    "Do not guess from indirect hints.\n"
)

# Maps every label spelling seen across stance datasets onto our fixed 3-way vocabulary.
STANCE_LABEL_MAP = {
    "FAVOR": "FAVOR",
    "AGAINST": "AGAINST",
    "NONE": "NONE",
    "PRO": "FAVOR",
    "NEUTRAL": "NONE",
    "UNCLEAR": "NONE",
    "UNRELATED": "NONE",
    "SUPPORTS": "FAVOR",
    "DENIES": "AGAINST",
}


def canonicalize_label(raw_label):
    normalized = str(raw_label).strip().upper()
    if normalized not in STANCE_LABEL_MAP:
        raise ValueError(
            f"Unrecognized stance label {raw_label!r} -- add it to STANCE_LABEL_MAP "
            f"(known: {sorted(STANCE_LABEL_MAP)})"
        )
    return STANCE_LABEL_MAP[normalized]


def build_conversations(df, text_column, target_column, label_column):
    conversations = []
    for _, row in df.iterrows():
        prompt = STANCE_PROMPT_TEMPLATE.format(target=row[target_column], text=row[text_column])
        answer = json.dumps({"stance": canonicalize_label(row[label_column])})
        conversations.append(
            [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer},
            ]
        )
    return conversations


## 2. Load the raw test files

1249 individual JSON files, one example per file: `_id`, `content`, `query`,
`stance_label`, `source_dataset`.


In [2]:
import json
from pathlib import Path

import pandas as pd

TEST_DIR = Path(
    "/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper"
    "/data_in/semeval/semeval2016_task6_testdata_gold"
)

records = [json.loads(p.read_text()) for p in sorted(TEST_DIR.glob("*.json"))]
df = pd.DataFrame(records)
print(f"{len(df)} test examples")
df.head()


1249 test examples


,_id,content,query,stance_label,source_dataset
0,10001,He who exalts himself shall be humbled; and he...,Atheism,AGAINST,SemEval-2016 Task 6 subtaskA testdata gold
1,10002,RT @prayerbullets: I remove Nehushtan -previou...,Atheism,AGAINST,SemEval-2016 Task 6 subtaskA testdata gold
2,10003,@Brainman365 @heidtjj @BenjaminLives I have so...,Atheism,AGAINST,SemEval-2016 Task 6 subtaskA testdata gold
3,10004,#God is utterly powerless without Human interv...,Atheism,AGAINST,SemEval-2016 Task 6 subtaskA testdata gold
4,10005,@David_Cameron Miracles of #Multiculturalism...,Atheism,AGAINST,SemEval-2016 Task 6 subtaskA testdata gold


## 3. Check the label distribution


In [3]:
df["stance_label"].value_counts()


stance_label
AGAINST    715
FAVOR      304
NONE       230
Name: count, dtype: int64

## 4. Build the `conversations` column


In [4]:
conversations = build_conversations(
    df,
    text_column="content",
    target_column="query",
    label_column="stance_label",
)
print(f"Built {len(conversations)} conversation examples")
conversations[0]


Built 1249 conversation examples


[{'role': 'user',
  'content': 'Stance classification is the task of determining the expressed or implied opinion, or stance, of a document toward a certain, specified target. Analyze the following document and determine its stance toward the provided query.\n\nQUERY: Atheism\n\nDOCUMENT: He who exalts himself shall be humbled; and he who humbles himself shall be exalted.Matt 23:12.     #SemST\n\nReturn valid JSON in exactly this format: {"stance": "FAVOR"}\nThe "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\nUse "FAVOR" only when the author is definitely in favor of the query. Use "AGAINST" only when the author is definitely against the query. Use "NONE" if any of the following holds: (a) the document does not discuss the query at all, (b) the document discusses it but the author takes no clear side (neutral/balanced), or (c) the author\'s position cannot be determined with confidence. Do not guess from indirect hints.\n'},
 {'role': 'assistant', 'content': '{"stan

## 5. Convert to a Hugging Face `Dataset` and sanity-check


In [5]:
from datasets import Dataset

test_dataset = Dataset.from_dict({"conversations": conversations})
assert "conversations" in test_dataset.column_names
print("Format OK")
test_dataset


/Users/nityaakalra/Desktop/fine_tune/fine-tuning-slms/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nityaakalra/Desktop/fine_tune/fine-tuning-slms/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Format OK


Dataset({
    features: ['conversations'],
    num_rows: 1249
})

## 6. Push to the Hugging Face Hub

Kept as a separate repo from `nityaak/semeval-stance-conversations` -- this is eval
data, never meant to be trained on.


In [7]:
PUSH_TO_HUB_ID = "nityaak/semeval-stance-test-gold"
PRIVATE = True

# Uncomment when ready:
test_dataset.push_to_hub(PUSH_TO_HUB_ID, private=PRIVATE)
print(f"Pushed to https://huggingface.co/datasets/{PUSH_TO_HUB_ID}")


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 363.24ba/s]
Processing Files (1 / 1): 100%|██████████|  178kB /  178kB, 17.1kB/s  
New Data Upload: 100%|██████████|  178kB /  178kB, 17.1kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.18s/ shards]


Pushed to https://huggingface.co/datasets/nityaak/semeval-stance-test-gold


## Recap

- Same `conversations` format as the training notebooks --
  `03_colab_hyperparameter_playground.ipynb`'s before/after check loads this directly
  once pushed
- This is the same held-out gold test set the full Snellius pipeline evaluates
  against (Tilburg vLLM) -- confirmed zero overlap with training data
- Push once, reuse everywhere -- no need to re-upload from your Mac each time
